In [73]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt

In [74]:
df = pd.read_csv("../annonces_reformulation_dataset.csv")

In [33]:
df

,input_text,tone,style,target_text
0,"iPhone 13 128 Go noir, batterie à 87 %, vendu ...",Professionnel,Concis,"iPhone 13 128 Go noir, batterie à 87 %, avec b..."
1,"Samsung Galaxy S21 256 Go violet, écran 6,2 po...",Chaleureux,Détaillé,Je vous propose avec plaisir un Samsung Galaxy...
2,"Tablette Lenovo Tab M10, écran 10,1 pouces, 64...",Neutre,Commercial,Tablette Lenovo Tab M10 à vendre : écran de 10...
3,"vds xiaomi redmi note 11 128go bleu tbe, 140e,...",Amical,Informatif,Marque : Xiaomi ; Modèle : Redmi Note 11 ; Sto...
4,"Google Pixel 6a 128 Go vert sauge, écran OLED ...",Persuasif,Descriptif,Le Google Pixel 6a se présente en coloris vert...
...,...,...,...,...
2983,"Ventilateur sur pied 50 W, diamètre 40 cm, min...",Amical,Concis,"Ventilateur sur pied de 50 W, diamètre 40 cm, ..."
2984,"Ventilateur sur pied 50 W, diamètre 40 cm, min...",Persuasif,Détaillé,Le ventilateur sur pied affiche une puissance ...
2985,"Ventilateur sur pied 50 W, diamètre 40 cm, min...",Enthousiaste,Commercial,"Ventilateur sur pied 50 W, diamètre 40 cm, min..."
2986,"Ventilateur sur pied 50 W, diamètre 40 cm, min...",Professionnel,Informatif,Type : ventilateur sur pied ; Puissance : 50 W...


In [34]:
df["input_text"].tolist()

["iPhone 13 128 Go noir, batterie à 87 %, vendu avec boîte et câble d'origine, 350 €. Je vends car je passe à un modèle plus récent.",
 "Samsung Galaxy S21 256 Go violet, écran 6,2 pouces, très bon état, pas de rayure sur l'écran, 280 euros.",
 'Tablette Lenovo Tab M10, écran 10,1 pouces, 64 Go de stockage, 3 Go de RAM, Wi-Fi uniquement, housse incluse, 120 €.',
 'vds xiaomi redmi note 11 128go bleu tbe, 140e, avec chargeur',
 'Google Pixel 6a 128 Go vert sauge, écran OLED 6,1 pouces, 6 Go de RAM, bon état général, 210 €.',
 'iPad Air 4e génération 64 Go gris sidéral, écran 10,9 pouces, Wi-Fi et cellulaire, vendu avec Apple Pencil 2e génération, 430 €.',
 "Oppo Reno 8 Lite 256 Go noir, 8 Go de RAM, écran 6,43 pouces, acheté il y a un an, état correct, boîte d'origine, 190 €.",
 'Samsung Galaxy Tab A8 32 Go gris, écran 10,5 pouces, 3 Go de RAM, avec coque de protection, 110 €.',
 "iPhone SE 2e génération 64 Go rouge, batterie à 91 %, écran sans casse, quelques micro-rayures au dos, 185 

In [35]:
df["target_text"].tolist()


["iPhone 13 128 Go noir, batterie à 87 %, avec boîte et câble d'origine, 350 €. Vente pour passer à un modèle plus récent.",
 "Je vous propose avec plaisir un Samsung Galaxy S21 en coloris violet. Sa capacité de stockage est de 256 Go. Son écran mesure 6,2 pouces et ne présente aucune rayure. L'appareil est en très bon état. Le prix demandé est de 280 euros.",
 'Tablette Lenovo Tab M10 à vendre : écran de 10,1 pouces, 64 Go de stockage, 3 Go de RAM, connexion Wi-Fi uniquement, housse incluse. Prix : 120 €. Contactez-moi pour toute question.',
 "Marque : Xiaomi ; Modèle : Redmi Note 11 ; Stockage : 128 Go ; Couleur : bleu ; État : très bon état ; Accessoire : chargeur ; Prix : 140 €. Si ça te dit, n'hésite pas à me contacter.",
 'Le Google Pixel 6a se présente en coloris vert sauge, avec un écran OLED de 6,1 pouces. Il dispose de 6 Go de RAM et de 128 Go de stockage. Son bon état général en fait une option solide à 210 €.',
 "iPad Air 4e génération 64 Go gris sidéral, écran 10,9 pouces,

In [75]:
%pip install -q transformers accelerate sentencepiece

import re
import difflib
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Trainer, TrainingArguments

MODEL_ID = "t5-small"
OUTPUT_DIR = Path("../modele_reformulation")
SEED = 42
MAX_LEN = 256
torch.manual_seed(SEED)

df = pd.read_csv("../annonces_reformulation_dataset.csv")

sources = df["input_text"].drop_duplicates().sample(frac=1, random_state=SEED).tolist()
n_val = round(len(sources) * 0.1)
val_sources = set(sources[:n_val])

train_df = df[~df["input_text"].isin(val_sources)].reset_index(drop=True)
val_df = df[df["input_text"].isin(val_sources)].reset_index(drop=True)
print("train :", len(train_df), "| validation :", len(val_df), "| annonces validation :", len(val_sources))


def make_input(row):
    return f"reformule en ton {row['tone']} et style {row['style']} : {row['input_text']}"

Note: you may need to restart the kernel to use updated packages.
train : 2700 | validation : 300 | annonces validation : 60


In [76]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


class ReformulationDataset(torch.utils.data.Dataset):
    def __init__(self, frame):
        inputs = [make_input(row) for _, row in frame.iterrows()]
        self.enc = tokenizer(inputs, max_length=MAX_LEN, truncation=True)
        self.labels = tokenizer(
            text_target=frame["target_text"].tolist(), max_length=MAX_LEN, truncation=True
        )["input_ids"]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return {
            "input_ids": self.enc["input_ids"][i],
            "attention_mask": self.enc["attention_mask"][i],
            "labels": self.labels[i],
        }


collate = DataCollatorForSeq2Seq(tokenizer, model=model)
train_ds = ReformulationDataset(train_df)
val_ds = ReformulationDataset(val_df)

print("cibles train contenant des caractères inconnus :", sum(tokenizer.unk_token_id in ids for ids in train_ds.labels))

c:\Users\WIN11\Desktop\Test_weli\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\WIN11\.cache\huggingface\hub\models--t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 131/131 [00:00<00:00, 315.63it/s]


cibles train contenant des caractères inconnus : 57


In [ ]:
args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=3e-4,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    warmup_steps=50,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=False,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate,
)
trainer.train()

model.save_pretrained(OUTPUT_DIR / "modele_final")
tokenizer.save_pretrained(OUTPUT_DIR / "modele_final")

In [ ]:
trainer.train()
model.save_pretrained(OUTPUT_DIR / "modele_final")
tokenizer.save_pretrained(OUTPUT_DIR / "modele_final")

In [ ]:
NUM_RE = re.compile(r"\d+(?:[.,]\d+)?")


def normalize(text):
    return re.sub(r"\s+", "", text.lower())


def digits_kept(source, output):
    return all(n in normalize(output) for n in NUM_RE.findall(source))


def is_copy(source, output):
    return difflib.SequenceMatcher(None, normalize(source), normalize(output)).ratio() >= 0.85


@torch.no_grad()
def generate(frame, batch_size=16):
    model.eval()
    inputs = [make_input(row) for _, row in frame.iterrows()]
    outputs = []
    for start in range(0, len(inputs), batch_size):
        enc = tokenizer(
            inputs[start:start + batch_size],
            max_length=MAX_LEN,
            truncation=True,
            padding=True,
            return_tensors="pt",
        ).to(model.device)
        out = model.generate(**enc, max_new_tokens=MAX_LEN, num_beams=4, early_stopping=True)
        outputs += tokenizer.batch_decode(out, skip_special_tokens=True)
    return [o.strip() for o in outputs]


val_df["prediction"] = generate(val_df)
val_df["chiffres_conserves"] = [digits_kept(s, p) for s, p in zip(val_df["input_text"], val_df["prediction"])]
val_df["quasi_copie"] = [is_copy(s, p) for s, p in zip(val_df["input_text"], val_df["prediction"])]

print("Chiffres conservés (prédictions) :", round(val_df["chiffres_conserves"].mean(), 3))
print("Quasi-copies (prédictions) :", round(val_df["quasi_copie"].mean(), 3))
print(
    "Chiffres conservés (références) :",
    round(sum(digits_kept(s, t) for s, t in zip(val_df["input_text"], val_df["target_text"])) / len(val_df), 3),
)

val_df.to_csv(OUTPUT_DIR / "predictions_validation.csv", index=False, encoding="utf-8")
val_df[["input_text", "tone", "style", "target_text", "prediction"]].head(5)